# 制造业PMI_经济增长：Hamilton Cycle 分段递推

本 notebook 使用 `output_宏观_pkl/M0017126.pkl` 的制造业 PMI，先用 Hamilton 回归法提取周期项 `hamilton_cycle`，再按固定长度分段。每一段会：

- 单独放入两状态区制切换模型，得到本段高/低增长状态概率；
- 将本段估计出的均值、扰动、状态转移概率和段末状态概率，作为下一段估计的初值；
- 同时用本段参数直接测量下一段，形成跨段递推检验。

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd


ROOT = Path.cwd()
PKL_DIR = ROOT / "output_宏观_pkl"
OUT = ROOT / "manufacturing_pmi_hamcycle_segment_recursive_output"
OUT.mkdir(exist_ok=True)

START_DATE = pd.Timestamp("2026-05-31")
PMI_CODE = "M0017126"
PMI_NAME = "中国:制造业PMI"

# 分段参数：60 个月约等于 5 年。可改为 36、48、72 做敏感性测试。
SEGMENT_MONTHS = 60
MIN_SEGMENT_OBS = 24

# Hamilton 参数：h 越小越敏感；p 是滞后阶数。
HAMILTON_H = 8
HAMILTON_P = 4

# 模型敏感度参数。
TRANSITION_STAY_INIT = 0.80
MIN_SWITCH_PROB = 0.08
LIKELIHOOD_SIGMA_SCALE = 0.85
STATE_PROB_THRESHOLD = 0.50
MAX_ITER = 500
TOL = 1e-8


## 读取 PMI 并生成 Hamilton 周期项

In [2]:
def _first_numeric_series(obj) -> pd.Series:
    if isinstance(obj, pd.Series):
        return pd.to_numeric(obj, errors="coerce")
    if isinstance(obj, pd.DataFrame):
        numeric = obj.apply(pd.to_numeric, errors="coerce")
        cols = [col for col in numeric.columns if numeric[col].notna().any()]
        if not cols:
            raise ValueError("DataFrame 中没有可转成数值的列")
        return numeric[cols[0]]
    return pd.to_numeric(pd.Series(obj), errors="coerce")


def read_monthly_pkl(code: str, start_date: pd.Timestamp = START_DATE) -> pd.Series:
    path = PKL_DIR / f"{code}.pkl"
    if not path.exists():
        raise FileNotFoundError(path)

    raw = _first_numeric_series(pd.read_pickle(path)).dropna()
    idx_text = pd.Series(raw.index.astype(str)).str.strip()
    parsed_idx = pd.to_datetime(idx_text, errors="coerce")

    if parsed_idx.notna().any():
        s = pd.Series(raw.to_numpy(), index=parsed_idx)
        s = s[s.index.notna()]
    else:
        dates = [start_date - pd.offsets.MonthEnd(i) for i in range(len(raw))]
        s = pd.Series(raw.to_numpy(), index=pd.DatetimeIndex(dates))

    month_end = pd.DatetimeIndex(s.index).to_period("M").to_timestamp("M")
    monthly = pd.Series(s.to_numpy(), index=month_end).sort_index().groupby(level=0).last()
    regular = monthly.reindex(pd.date_range(monthly.index.min(), monthly.index.max(), freq="ME"))
    return regular.interpolate(limit=2).dropna().rename(code)


def hamilton_filter(y: pd.Series, h: int = HAMILTON_H, p: int = HAMILTON_P) -> tuple[pd.Series, pd.Series]:
    x = y.astype(float).dropna()
    rows = []
    targets = []
    dates = []
    for t in range(p - 1, len(x) - h):
        rows.append([1.0] + [x.iloc[t - lag] for lag in range(p)])
        targets.append(x.iloc[t + h])
        dates.append(x.index[t + h])

    if len(rows) <= p:
        raise ValueError("Hamilton 回归样本不足")

    xmat = np.asarray(rows, dtype=float)
    yvec = np.asarray(targets, dtype=float)
    beta = np.linalg.lstsq(xmat, yvec, rcond=None)[0]
    trend = pd.Series(xmat @ beta, index=pd.DatetimeIndex(dates), name="hamilton_trend")
    cycle = (x.reindex(trend.index) - trend).rename("hamilton_cycle")
    return cycle, trend


raw_pmi = read_monthly_pkl(PMI_CODE)
ham_cycle, ham_trend = hamilton_filter(raw_pmi)

ham_data = pd.concat(
    {
        "raw_pmi": raw_pmi,
        "hamilton_trend": ham_trend,
        "hamilton_cycle": ham_cycle,
    },
    axis=1,
).rename_axis("date")

display(pd.DataFrame([{
    "code": PMI_CODE,
    "name": PMI_NAME,
    "raw_start": raw_pmi.index.min(),
    "raw_end": raw_pmi.index.max(),
    "ham_cycle_start": ham_cycle.index.min(),
    "ham_cycle_end": ham_cycle.index.max(),
    "ham_cycle_nobs": len(ham_cycle),
    "segment_months": SEGMENT_MONTHS,
}]))
display(ham_data.tail(12))


C:\Users\16492\AppData\Local\Temp\ipykernel_29120\3520083.py:20: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed_idx = pd.to_datetime(idx_text, errors="coerce")


,code,name,raw_start,raw_end,ham_cycle_start,ham_cycle_end,ham_cycle_nobs,segment_months
0,M0017126,中国:制造业PMI,2005-01-31,2026-05-31,2005-12-31,2026-05-31,246,60


,raw_pmi,hamilton_trend,hamilton_cycle
date,,,
2025-06-30,49.7,50.668511,-0.968511
2025-07-31,49.3,50.642530,-1.342530
2025-08-31,49.4,50.713396,-1.313396
2025-09-30,49.8,50.619295,-0.819295
2025-10-31,49.0,50.772688,-1.772688
2025-11-30,49.2,50.826826,-1.626826
2025-12-31,50.1,50.475324,-0.375324
2026-01-31,49.3,50.658127,-1.358127
2026-02-28,49.0,50.768930,-1.768930


## 两状态模型函数

In [3]:
def _normal_pdf(values: np.ndarray, mu: np.ndarray, sigma: float) -> np.ndarray:
    sigma = max(float(sigma), 1e-8)
    z = (values[:, None] - mu[None, :]) / sigma
    return np.exp(-0.5 * z * z) / (np.sqrt(2.0 * np.pi) * sigma)


def _constrain_transition(transition: np.ndarray, min_switch_prob: float) -> np.ndarray:
    transition = np.asarray(transition, dtype=float).copy()
    min_switch_prob = float(np.clip(min_switch_prob, 0.0, 0.49))
    max_stay_prob = 1.0 - min_switch_prob
    for i in range(transition.shape[0]):
        transition[i, i] = min(transition[i, i], max_stay_prob)
        off_diag = [j for j in range(transition.shape[1]) if j != i]
        remaining = 1.0 - transition[i, i]
        off_sum = transition[i, off_diag].sum()
        if off_sum <= 0:
            transition[i, off_diag] = remaining / len(off_diag)
        else:
            transition[i, off_diag] = transition[i, off_diag] / off_sum * remaining
    return transition / transition.sum(axis=1, keepdims=True)


def _params_to_arrays(initial_params: dict | None, values: np.ndarray) -> tuple[np.ndarray, float, np.ndarray, np.ndarray]:
    if initial_params is None:
        q25, q75 = np.nanpercentile(values, [25, 75])
        mu = np.asarray([q25, q75], dtype=float)
        sigma = float(np.nanstd(values, ddof=1)) or 1.0
        stay = float(np.clip(TRANSITION_STAY_INIT, 0.51, 0.99))
        transition = np.asarray([[stay, 1.0 - stay], [1.0 - stay, stay]], dtype=float)
        init_prob = np.asarray([0.50, 0.50], dtype=float)
        return mu, sigma, _constrain_transition(transition, MIN_SWITCH_PROB), init_prob

    mu = np.asarray([initial_params["mu_low"], initial_params["mu_high"]], dtype=float)
    sigma = float(initial_params.get("sigma", np.nanstd(values, ddof=1) or 1.0))
    transition = np.asarray(
        [
            [initial_params["p_stay_low"], initial_params["p_switch_low_to_high"]],
            [initial_params["p_switch_high_to_low"], initial_params["p_stay_high"]],
        ],
        dtype=float,
    )
    init_prob = np.asarray(initial_params.get("next_init_prob", [0.50, 0.50]), dtype=float)
    init_prob = init_prob / init_prob.sum()
    return mu, sigma, _constrain_transition(transition, MIN_SWITCH_PROB), init_prob


def _forward_filter(values: np.ndarray, mu: np.ndarray, sigma: float, transition: np.ndarray, init_prob: np.ndarray):
    emission = _normal_pdf(values, mu, sigma)
    prior = np.zeros((len(values), 2))
    filtered = np.zeros((len(values), 2))
    scale = np.zeros(len(values))

    prior[0] = init_prob
    filtered[0] = prior[0] * emission[0]
    scale[0] = filtered[0].sum()
    filtered[0] /= scale[0]

    for t in range(1, len(values)):
        prior[t] = filtered[t - 1] @ transition
        filtered[t] = prior[t] * emission[t]
        scale[t] = filtered[t].sum()
        filtered[t] /= scale[t]

    return prior, filtered, float(np.log(scale).sum())


def _state_frame(index: pd.DatetimeIndex, values: np.ndarray, prior: np.ndarray, filtered: np.ndarray) -> pd.DataFrame:
    out = pd.DataFrame(index=index)
    out["model_value"] = values
    out["prior_prob_low_state"] = prior[:, 0]
    out["prior_prob_high_state"] = prior[:, 1]
    out["prob_low_state"] = filtered[:, 0]
    out["prob_high_state"] = filtered[:, 1]
    out["state"] = np.where(out["prob_high_state"] >= STATE_PROB_THRESHOLD, "高均值状态", "低均值状态")
    out["growth_regime"] = np.where(out["prob_high_state"] >= STATE_PROB_THRESHOLD, "经济增长偏强", "经济增长偏弱")
    return out


def fit_segment_hmm(y: pd.Series, initial_params: dict | None = None):
    x = y.astype(float).replace([np.inf, -np.inf], np.nan).dropna()
    values = x.to_numpy()
    if len(values) < MIN_SEGMENT_OBS:
        raise ValueError(f"分段样本不足：{len(values)} < {MIN_SEGMENT_OBS}")

    mu, sigma, transition, init_prob = _params_to_arrays(initial_params, values)
    last_ll = -np.inf

    for iteration in range(1, MAX_ITER + 1):
        emission = _normal_pdf(values, mu, sigma)
        alpha = np.zeros((len(values), 2))
        scale = np.zeros(len(values))
        alpha[0] = init_prob * emission[0]
        scale[0] = alpha[0].sum()
        alpha[0] /= scale[0]

        for t in range(1, len(values)):
            alpha[t] = (alpha[t - 1] @ transition) * emission[t]
            scale[t] = alpha[t].sum()
            alpha[t] /= scale[t]

        beta = np.ones((len(values), 2))
        for t in range(len(values) - 2, -1, -1):
            beta[t] = transition @ (emission[t + 1] * beta[t + 1])
            beta[t] /= scale[t + 1]

        gamma = alpha * beta
        gamma /= gamma.sum(axis=1, keepdims=True)

        xi_sum = np.zeros((2, 2))
        for t in range(len(values) - 1):
            xi = alpha[t, :, None] * transition * emission[t + 1, None, :] * beta[t + 1, None, :]
            xi_sum += xi / xi.sum()

        init_prob = gamma[0]
        transition = _constrain_transition(xi_sum / xi_sum.sum(axis=1, keepdims=True), MIN_SWITCH_PROB)
        weights = gamma.sum(axis=0)
        mu = (gamma * values[:, None]).sum(axis=0) / weights
        sigma = np.sqrt(((gamma * (values[:, None] - mu[None, :]) ** 2).sum()) / len(values))

        ll = float(np.log(scale).sum())
        if abs(ll - last_ll) < TOL:
            break
        last_ll = ll

    order = np.argsort(mu)
    mu = mu[order]
    transition = transition[np.ix_(order, order)]
    init_prob = init_prob[order]
    init_prob = init_prob / init_prob.sum()

    sigma_for_filter = max(float(sigma) * LIKELIHOOD_SIGMA_SCALE, 1e-8)
    prior, filtered, filtered_ll = _forward_filter(values, mu, sigma_for_filter, transition, init_prob)
    states = _state_frame(x.index, values, prior, filtered)

    next_init_prob = filtered[-1] @ transition
    params = {
        "mu_low": float(mu[0]),
        "mu_high": float(mu[1]),
        "sigma": float(sigma),
        "sigma_for_filter": float(sigma_for_filter),
        "p_stay_low": float(transition[0, 0]),
        "p_switch_low_to_high": float(transition[0, 1]),
        "p_switch_high_to_low": float(transition[1, 0]),
        "p_stay_high": float(transition[1, 1]),
        "init_prob_low": float(init_prob[0]),
        "init_prob_high": float(init_prob[1]),
        "end_prob_low": float(filtered[-1, 0]),
        "end_prob_high": float(filtered[-1, 1]),
        "next_init_prob": next_init_prob / next_init_prob.sum(),
        "next_init_prob_low": float(next_init_prob[0]),
        "next_init_prob_high": float(next_init_prob[1]),
        "log_likelihood": filtered_ll,
        "iterations": iteration,
        "nobs": len(values),
    }
    return states, params


def apply_params_to_segment(y: pd.Series, params: dict) -> pd.DataFrame:
    x = y.astype(float).replace([np.inf, -np.inf], np.nan).dropna()
    values = x.to_numpy()
    mu, sigma, transition, init_prob = _params_to_arrays(params, values)
    sigma_for_filter = max(float(sigma) * LIKELIHOOD_SIGMA_SCALE, 1e-8)
    prior, filtered, filtered_ll = _forward_filter(values, mu, sigma_for_filter, transition, init_prob)
    out = _state_frame(x.index, values, prior, filtered)
    out["log_likelihood_under_previous_params"] = filtered_ll
    return out


## 分段递推估计

In [4]:
def make_segments(series: pd.Series, segment_months: int = SEGMENT_MONTHS) -> list[pd.Series]:
    x = series.dropna().sort_index()
    segments = []
    for start in range(0, len(x), segment_months):
        part = x.iloc[start : start + segment_months]
        if len(part) >= MIN_SEGMENT_OBS:
            segments.append(part)
    return segments


segments = make_segments(ham_cycle)

current_state_tables = []
next_eval_tables = []
segment_rows = []
parameter_rows = []

previous_params = None
for i, segment in enumerate(segments, start=1):
    segment_id = f"segment_{i:02d}"
    init_source = "sample_quantiles" if previous_params is None else f"segment_{i - 1:02d}_params"
    states, params = fit_segment_hmm(segment, initial_params=previous_params)

    states = states.rename_axis("date").reset_index()
    states.insert(0, "segment_id", segment_id)
    states.insert(1, "evaluation_type", "current_segment_refit")
    current_state_tables.append(states)

    segment_rows.append({
        "segment_id": segment_id,
        "init_source": init_source,
        "start": segment.index.min(),
        "end": segment.index.max(),
        "months": len(segment),
        "latest_date": states["date"].iloc[-1],
        "latest_prob_high_state": states["prob_high_state"].iloc[-1],
        "latest_state": states["state"].iloc[-1],
        "latest_growth_regime": states["growth_regime"].iloc[-1],
        "state_switches": int(states["state"].ne(states["state"].shift()).sum() - 1),
        "avg_abs_prob_high_change": states["prob_high_state"].diff().abs().mean(),
    })

    parameter_rows.append({
        "segment_id": segment_id,
        "init_source": init_source,
        "start": segment.index.min(),
        "end": segment.index.max(),
        **{k: v for k, v in params.items() if k != "next_init_prob"},
    })

    if i < len(segments):
        next_segment = segments[i]
        next_eval = apply_params_to_segment(next_segment, params)
        next_eval = next_eval.rename_axis("date").reset_index()
        next_eval.insert(0, "source_segment_id", segment_id)
        next_eval.insert(1, "target_segment_id", f"segment_{i + 1:02d}")
        next_eval.insert(2, "evaluation_type", "next_segment_using_previous_params")
        next_eval_tables.append(next_eval)

    previous_params = params

current_states = pd.concat(current_state_tables, ignore_index=True)
next_segment_evaluation = pd.concat(next_eval_tables, ignore_index=True) if next_eval_tables else pd.DataFrame()
segment_summary = pd.DataFrame(segment_rows)
parameter_trace = pd.DataFrame(parameter_rows)
latest_state = segment_summary.tail(1).copy()

display(segment_summary)
display(parameter_trace)
display(next_segment_evaluation.tail(20))


,segment_id,init_source,start,end,months,latest_date,latest_prob_high_state,latest_state,latest_growth_regime,state_switches,avg_abs_prob_high_change
0,segment_01,sample_quantiles,2005-12-31,2010-11-30,60,2010-11-30,9.999973e-01,高均值状态,经济增长偏强,4,0.060991
1,segment_02,segment_01_params,2010-12-31,2015-11-30,60,2015-11-30,5.396385e-07,低均值状态,经济增长偏弱,5,0.087259
2,segment_03,segment_02_params,2015-12-31,2020-11-30,60,2020-11-30,9.686213e-01,高均值状态,经济增长偏强,3,0.064863
3,segment_04,segment_03_params,2020-12-31,2025-11-30,60,2025-11-30,5.420978e-05,低均值状态,经济增长偏弱,5,0.094808


,segment_id,init_source,start,end,mu_low,mu_high,sigma,sigma_for_filter,p_stay_low,p_switch_low_to_high,...,p_stay_high,init_prob_low,init_prob_high,end_prob_low,end_prob_high,next_init_prob_low,next_init_prob_high,log_likelihood,iterations,nobs
0,segment_01,sample_quantiles,2005-12-31,2010-11-30,-6.315343,2.840187,2.399595,2.039656,0.851920,0.148080,...,0.920000,7.064232e-37,1.000000e+00,0.000003,9.999973e-01,0.080002,0.919998,-148.182018,15,60
1,segment_02,segment_01_params,2010-12-31,2015-11-30,-0.592765,1.348672,0.585995,0.498096,0.920000,0.080000,...,0.700391,1.194018e-138,1.000000e+00,0.999999,5.396385e-07,0.920000,0.080000,-64.797290,40,60
2,segment_03,segment_02_params,2015-12-31,2020-11-30,-1.416350,0.334211,1.940379,1.649322,0.879414,0.120586,...,0.920000,1.000000e+00,3.577203e-17,0.031379,9.686213e-01,0.105085,0.894915,-130.328983,33,60
3,segment_04,segment_03_params,2020-12-31,2025-11-30,-1.180963,0.893298,0.759522,0.645594,0.920000,0.080000,...,0.683027,1.710027e-53,1.000000e+00,0.999946,5.420978e-05,0.919967,0.080033,-79.156757,24,60


,source_segment_id,target_segment_id,evaluation_type,date,model_value,prior_prob_low_state,prior_prob_high_state,prob_low_state,prob_high_state,state,growth_regime,log_likelihood_under_previous_params
160,segment_03,segment_04,next_segment_using_previous_params,2024-04-30,-0.124495,0.612930,0.387070,0.547746,0.452254,低均值状态,经济增长偏弱,-100.607724
161,segment_03,segment_04,next_segment_using_previous_params,2024-05-31,-1.118386,0.517876,0.482124,0.608986,0.391014,低均值状态,经济增长偏弱,-100.607724
162,segment_03,segment_04,next_segment_using_previous_params,2024-06-30,-1.072502,0.566832,0.433168,0.648153,0.351847,低均值状态,经济增长偏弱,-100.607724
163,segment_03,segment_04,next_segment_using_previous_params,2024-07-31,-1.191235,0.598143,0.401857,0.693414,0.306586,低均值状态,经济增长偏弱,-100.607724
164,segment_03,segment_04,next_segment_using_previous_params,2024-08-31,-1.523909,0.634326,0.365674,0.765541,0.234459,低均值状态,经济增长偏弱,-100.607724
165,segment_03,segment_04,next_segment_using_previous_params,2024-09-30,-0.744650,0.691985,0.308015,0.719185,0.280815,低均值状态,经济增长偏弱,-100.607724
166,segment_03,segment_04,next_segment_using_previous_params,2024-10-31,-0.430443,0.654927,0.345073,0.638665,0.361335,低均值状态,经济增长偏弱,-100.607724
167,segment_03,segment_04,next_segment_using_previous_params,2024-11-30,-0.385330,0.590558,0.409442,0.566123,0.433877,低均值状态,经济增长偏弱,-100.607724
168,segment_03,segment_04,next_segment_using_previous_params,2024-12-31,-0.598857,0.532567,0.467433,0.541812,0.458188,低均值状态,经济增长偏弱,-100.607724
169,segment_03,segment_04,next_segment_using_previous_params,2025-01-31,-1.424308,0.513132,0.486868,0.650430,0.349570,低均值状态,经济增长偏弱,-100.607724


## 导出结果

In [5]:
ham_data.to_csv(OUT / "pmi_hamilton_cycle_series.csv", encoding="utf-8-sig")
segment_summary.to_csv(OUT / "pmi_hamcycle_segment_summary.csv", index=False, encoding="utf-8-sig")
parameter_trace.to_csv(OUT / "pmi_hamcycle_parameter_trace.csv", index=False, encoding="utf-8-sig")
current_states.to_csv(OUT / "pmi_hamcycle_current_segment_states.csv", index=False, encoding="utf-8-sig")
next_segment_evaluation.to_csv(OUT / "pmi_hamcycle_next_segment_evaluation.csv", index=False, encoding="utf-8-sig")
latest_state.to_csv(OUT / "pmi_hamcycle_latest_state.csv", index=False, encoding="utf-8-sig")

excel_path = OUT / "pmi_hamcycle_segment_recursive.xlsx"
with pd.ExcelWriter(excel_path) as writer:
    ham_data.reset_index().to_excel(writer, sheet_name="ham_cycle_series", index=False)
    segment_summary.to_excel(writer, sheet_name="segment_summary", index=False)
    parameter_trace.to_excel(writer, sheet_name="parameter_trace", index=False)
    current_states.to_excel(writer, sheet_name="current_segment_states", index=False)
    next_segment_evaluation.to_excel(writer, sheet_name="next_segment_eval", index=False)
    latest_state.to_excel(writer, sheet_name="latest_state", index=False)

print(f"已导出到：{OUT}")
print(f"Excel 文件：{excel_path}")


已导出到：c:\Users\16492\Desktop\实习内容\数据处理\data\manufacturing_pmi_hamcycle_segment_recursive_output
Excel 文件：c:\Users\16492\Desktop\实习内容\数据处理\data\manufacturing_pmi_hamcycle_segment_recursive_output\pmi_hamcycle_segment_recursive.xlsx
